# Validate the access-profile inputs

**Inherited contract:** the three immutable matrices published by the pinned PCADI reference release.

**Purpose:** confirm that the modelling inputs satisfy their complete contracts before any model is fitted.

The gate checks the exact columns and order, identifier integrity, dimensions, feature domains, checksums, byte sizes, separate one-day and two-to-seven-day booking features, cohort nesting, inherited national values and the pinned PCADI source commit.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CONFIG_PATH = ROOT / 'configs' / 'reference_apr2025_mar2026.json'
from gpap2.config import load_config
REFERENCE_CONFIG = load_config(CONFIG_PATH)
AUTHORITY_MANIFEST = REFERENCE_CONFIG.resolve(REFERENCE_CONFIG.authority_checksum_file)
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.precision', 6)


## Method contract

This stage inherits the three PCADI matrices and checks their identity, provenance, dimensions, feature order, value domains and cohort relationships. The tables below are generated from the loaded [reference configuration](../configs/reference_apr2025_mar2026.json), [PCADI input contracts](../data/contracts/pcadi_input_contracts.csv), [validation implementation](../src/gpap2/validation.py) and [notebook reporting helpers](../src/gpap2/notebook_reporting.py). They do not reproduce PCADI's source-integration pipeline.

In [2]:
from gpap2.config import load_config
from gpap2.io import read_contract_csv
from gpap2.notebook_reporting import (
    build_input_contract_table,
    build_output_contract_table,
    build_quality_gate_table,
)
from gpap2.validation import validate_cohort_relationships, validate_contract_directory

config = load_config(CONFIG_PATH)
contracts = read_contract_csv(config.resolve(config.contracts_file))
matrix_checks = validate_contract_directory(
    config.resolve(config.input_directory),
    config.resolve(config.contracts_file),
    config,
)
cohort_checks = validate_cohort_relationships(config.resolve(config.input_directory), config)
input_contract = build_input_contract_table(config, contracts, matrix_checks)
input_contract

,public_filename,analytical_role,observation_period,rows,numeric_features,identifier,feature_names_in_order,sha256,upstream_repository,upstream_commit,upstream_tag
0,primary_practice_access_clustering_matrix.csv,national_primary,2025-04-01 to 2026-03-31,6067,14,practice_code_standardised,ocs_submissions_per_1000_patient_months | ocs_...,C50B14AA191C54C29201DC9909E138395C1A2AEA7F596E...,PeterHDS/pcadi-data-integration,1239c63356acfb824277ee6fbaee25fa8df51313,reference-apr2025-mar2026
1,cbt_inbound_sensitivity_clustering_matrix_17_f...,telephone_inbound_sensitivity,2025-04-01 to 2026-03-31,3020,17,practice_code_standardised,ocs_submissions_per_1000_patient_months | ocs_...,CCC179B870BBD3EC46DD1B75868DB38156FE23A44BBC5A...,PeterHDS/pcadi-data-integration,1239c63356acfb824277ee6fbaee25fa8df51313,reference-apr2025-mar2026
2,cbt_outcomes_sensitivity_clustering_matrix_21_...,telephone_outcome_source_representation,2025-04-01 to 2026-03-31,1456,21,practice_code_standardised,ocs_submissions_per_1000_patient_months | ocs_...,D3D2E70C1A718260DD332B59F835EB6316826677A1DF5C...,PeterHDS/pcadi-data-integration,1239c63356acfb824277ee6fbaee25fa8df51313,reference-apr2025-mar2026


In [3]:
quality_gates = build_quality_gate_table(matrix_checks, cohort_checks)
quality_gates

,scope,quality_gate,expected,observed,passed
0,all three PCADI matrices,blank identifiers,0,primary_practice_access_clustering_matrix=0 | ...,True
1,all three PCADI matrices,duplicate identifiers,0,primary_practice_access_clustering_matrix=0 | ...,True
2,all three PCADI matrices,missing numerical values,0,primary_practice_access_clustering_matrix=0 | ...,True
3,all three PCADI matrices,non-numeric values,0,primary_practice_access_clustering_matrix=0 | ...,True
4,all three PCADI matrices,non-finite values,0,primary_practice_access_clustering_matrix=0 | ...,True
5,all three PCADI matrices,negative values,0,primary_practice_access_clustering_matrix=0 | ...,True
6,all three PCADI matrices,shares outside permitted range,0,primary_practice_access_clustering_matrix=0 | ...,True
7,all three PCADI matrices,exact feature names and order,True,primary_practice_access_clustering_matrix=True...,True
8,all three PCADI matrices,separate one-day and two-to-seven-day features,True,primary_practice_access_clustering_matrix=True...,True
9,all three PCADI matrices,obsolete combined booking band absent,True,primary_practice_access_clustering_matrix=True...,True


In [4]:
assert matrix_checks['passed'].all(), matrix_checks.loc[~matrix_checks['passed'], ['filename', 'failure_reasons']]
assert cohort_checks['passed'].all(), cohort_checks.loc[~cohort_checks['passed']]
assert quality_gates['passed'].all(), quality_gates.loc[~quality_gates['passed']]
output_contract = build_output_contract_table({
    'validated matrices': int(matrix_checks['passed'].sum()),
    'validated cohort relationships': int(cohort_checks['passed'].sum()),
    'failed gates': int((~quality_gates['passed']).sum()),
    'handover': 'fixed national matrix and feature contract',
})
output_contract

,measure,observed
0,validated matrices,3
1,validated cohort relationships,4
2,failed gates,0
3,handover,fixed national matrix and feature contract


## Decision

The checksum-controlled PCADI outputs can enter GPAP² without recalculation or imputation. The practice identifier is retained for traceability and excluded from every numerical model.

**Stage handover:** The validated matrix and feature contract become the fixed input to national profile modelling.